In [1]:
%reload_ext autoreload
%autoreload 2

# Klebsiella EBI AMR records
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

file = '/home/dca36/rds/rds-floto-bacterial-4k08a2yyQLw/david/raw/klebsiella_ebi_amr_records_20260216.csv'
ast_data = pd.read_csv(file) # Comma separated values

# Print the first 5 rows of the dataframe
display(ast_data.head())



/tmp/ipykernel_2406531/47765919.py:12: DtypeWarning: Columns (16,20,21,22) have mixed types. Specify dtype option on import or set low_memory=False.
  ast_data = pd.read_csv(file) # Comma separated values


,phenotype-antibiotic_name,phenotype-resistance_phenotype,phenotype-gen_measurement,phenotype-ast_standard,phenotype-laboratory_typing_method,phenotype-platform,phenotype-BioSample_ID,phenotype-assembly_ID,phenotype-genus,phenotype-species,...,phenotype-isolation_latitude,phenotype-isolation_longitude,phenotype-AMR_associated_publications,phenotype-Updated_phenotype_CLSI,phenotype-Updated_phenotype_EUCAST,phenotype-used_ECOFF,phenotype-database,phenotype-country,phenotype-geographical_region,phenotype-geographical_subregion
0,ceftriaxone,resistant,> 32 mg/L,CLSI,broth dilution,BD Phoenix,SAMN06437146,GCA_022066305.1,Klebsiella,Klebsiella pneumoniae,...,29.710689,-95.400586,29323230;33075053;33659219,resistant,resistant,no,PATRIC;CABBAGE_PubMed_data,United States of America,Americas,Northern America
1,cefuroxime,resistant,> 16 mg/L,CLSI,broth dilution,BD Phoenix,SAMN06437146,GCA_022066305.1,Klebsiella,Klebsiella pneumoniae,...,29.710689,-95.400586,29323230;33075053;33659219,resistant,resistant,no,PATRIC;CABBAGE_PubMed_data,United States of America,Americas,Northern America
2,levofloxacin,resistant,> 4 mg/L,CLSI,broth dilution,BD Phoenix,SAMN06437146,GCA_022066305.1,Klebsiella,Klebsiella pneumoniae,...,29.710689,-95.400586,29323230;33075053;33659219,resistant,resistant,no,PATRIC;CABBAGE_PubMed_data,United States of America,Americas,Northern America
3,gentamicin,resistant,> 8 mg/L,CLSI,broth dilution,BD Phoenix,SAMN06437146,GCA_022066305.1,Klebsiella,Klebsiella pneumoniae,...,29.710689,-95.400586,29323230;33075053;33659219,resistant,resistant,no,PATRIC;CABBAGE_PubMed_data,United States of America,Americas,Northern America
4,imipenem,resistant,> 8 mg/L,CLSI,broth dilution,BD Phoenix,SAMN06437146,GCA_022066305.1,Klebsiella,Klebsiella pneumoniae,...,29.710689,-95.400586,29323230;33075053;33659219,resistant,resistant,no,PATRIC;CABBAGE_PubMed_data,United States of America,Americas,Northern America


In [2]:
# Create a binary column for "Susceptible" and "Resistant"
ebi_mic_column = "phenotype-gen_measurement"
import re

def parse_number_section(number_section):
    """Parse the 'number' part of a MIC string (e.g. '32', '.25', '64/4').
    - Takes anything up to '/' if present and discards the rest.
    - If the number starts with '.', prepends '0' for clarity.
    Returns float or None on invalid input.
    """
    if not isinstance(number_section, str) or not number_section.strip():
        return None
    # 1. Take only the part before '/' if present
    num_str = number_section.split("/")[0].strip()
    if not num_str:
        return None
    # 2. Leading dot: treat as 0.xxx
    if num_str.startswith(".") and (len(num_str) == 1 or num_str[1].isdigit() or num_str[1] == "."):
        num_str = "0" + num_str
    try:
        return float(num_str)
    except ValueError:
        return None

# Regex for the number section only (used to validate before parsing).
#   ^           = start of string
#   \d*         = zero or more digits
#   \.?         = optional literal dot (\. is dot, ? means 0 or 1)
#   \d+         = one or more digits (at least one digit somewhere)
#   (?:/\d+.*)? = optional non-capturing group (?: ... )? ; \/ = slash ; \d+ = digits ; .* = rest
#   $           = end of string
_number_section_pattern = re.compile(r"^\d*\.?\d+(?:/\d+.*)?$")

VALID_OPERATORS = {">", "<", ">=", "<=", "=", "=="}

def convert_ebi_mic_data(ast_data, ebi_mic_column="phenotype-gen_measurement"):
    """Convert EBI MIC data to log-scale MIC values with adjustments for inequality operators.

    Parses MIC by splitting on space: operator, then number section, then optional units.
    Creates log-transformed MIC with adjustments for censored data:
    - For "<" values: log_mic is reduced by 1 (equivalent to dividing MIC by 10)
    - For ">" values: log_mic is increased by 1 (equivalent to multiplying MIC by 10)
    - For other operators (=, ==, <=, >=): log_mic uses the exact value

    Parameters
    ----------
    ast_data : pandas.DataFrame
        DataFrame containing AST (Antimicrobial Susceptibility Testing) data
    ebi_mic_column : str, default="phenotype-gen_measurement"
        Column name containing MIC values in format "operator number_section units"
        (space-separated; number_section can be e.g. "32", ".25", "64/4")

    Returns
    -------
    tuple of (pandas.DataFrame, list of dict)
        - DataFrame with added columns MIC_meaning, MIC_value, log_mic (NA where unparsable)
        - List of unparsable results: [{"index": idx, "raw": str, "reason": str}, ...]
    """
    nan_mask = ast_data[ebi_mic_column].isna()
    n_nan = nan_mask.sum()
    if n_nan > 0:
        print(f"Warning: Found {n_nan} NaN values in {ebi_mic_column}, these will be excluded from parsing")

    ast_data["MIC_meaning"] = pd.NA
    ast_data["MIC_value"] = pd.NA
    ast_data["log_mic"] = pd.NA

    valid_data = ast_data[~nan_mask].copy()
    if len(valid_data) == 0:
        print("Warning: No valid (non-NaN) values to parse")
        return ast_data, []

    unparsable = []
    mic_meaning = []
    mic_value = []
    log_mic = []

    for idx, row in valid_data.iterrows():
        raw = row[ebi_mic_column]
        parts = raw.strip().split()
        if len(parts) < 2:
            unparsable.append({"index": idx, "raw": raw, "reason": "fewer than 2 space-separated parts"})
            mic_meaning.append(pd.NA)
            mic_value.append(np.nan)
            log_mic.append(np.nan)
            continue
        operator, number_section = parts[0], parts[1]
        if operator not in VALID_OPERATORS:
            unparsable.append({"index": idx, "raw": raw, "reason": f"invalid operator {operator!r}"})
            mic_meaning.append(pd.NA)
            mic_value.append(np.nan)
            log_mic.append(np.nan)
            continue
        if not _number_section_pattern.match(number_section):
            unparsable.append({"index": idx, "raw": raw, "reason": f"number section does not match expected pattern: {number_section!r}"})
            mic_meaning.append(pd.NA)
            mic_value.append(np.nan)
            log_mic.append(np.nan)
            continue
        value = parse_number_section(number_section)
        if value is None:
            unparsable.append({"index": idx, "raw": raw, "reason": f"parse_number_section failed for {number_section!r}"})
            mic_meaning.append(pd.NA)
            mic_value.append(np.nan)
            log_mic.append(np.nan)
            continue
        mic_meaning.append(operator)
        mic_value.append(value)
        log_mic.append(np.log10(value))

    valid_data["MIC_meaning"] = mic_meaning
    valid_data["MIC_value"] = mic_value
    valid_data["log_mic"] = np.array(log_mic, dtype=float)
    valid_data.loc[valid_data["MIC_meaning"] == "<", "log_mic"] -= 1
    valid_data.loc[valid_data["MIC_meaning"] == ">", "log_mic"] += 1

    ast_data.loc[~nan_mask, "MIC_meaning"] = valid_data["MIC_meaning"]
    ast_data.loc[~nan_mask, "MIC_value"] = valid_data["MIC_value"]
    ast_data.loc[~nan_mask, "log_mic"] = valid_data["log_mic"]

    n_ok = (~valid_data["MIC_value"].isna()).sum()
    print(f"\nSuccessfully parsed {n_ok} MIC values")
    if unparsable:
        print(f"Unparsable: {len(unparsable)} rows (returned in second value for inspection)")
    print("Operator distribution:")
    print(ast_data["MIC_meaning"].value_counts())

    return ast_data, unparsable


ast_data, unparsable = convert_ebi_mic_data(ast_data)
if unparsable:
    print(f"\nUnparsable rows ({len(unparsable)}):")
    for u in unparsable:
        print(f"  index={u['index']!r} raw={u['raw']!r} reason={u['reason']}")



Successfully parsed 106211 MIC values
Operator distribution:
MIC_meaning
>     40517
<=    25469
=     20546
>=    10913
==     8156
<       610
Name: count, dtype: int64


In [3]:
# Lets get a column Sample_ID = "phentoype-BioSample_ID"
ast_data['Sample_ID'] = ast_data['phenotype-BioSample_ID']

# How many unique Klebsiella Genomes Sample_IDs are there?
print(f"Number of unique Klebsiella Genomes: {ast_data['Sample_ID'].nunique()}")

# How many unique antibiotics are tested (use nunique on column "phenotype-antibiotic_name")
unique_antibiotics = ast_data['phenotype-antibiotic_name'].nunique()
# Order descending and display all with > 1000 records
ast_data_antibiotics = ast_data.groupby('phenotype-antibiotic_name')['phenotype-antibiotic_name'].count().sort_values(ascending=False)
display(ast_data_antibiotics[ast_data_antibiotics > 1000])

Number of unique Klebsiella Genomes: 10260


phenotype-antibiotic_name
meropenem                        8086
gentamicin                       6837
ceftazidime                      6757
ciprofloxacin                    6729
amikacin                         5682
trimethoprim-sulfamethoxazole    5639
piperacillin-tazobactam          5348
cefepime                         4750
ceftriaxone                      4602
imipenem                         4580
tobramycin                       4351
ampicillin                       4223
aztreonam                        4171
cefoxitin                        3995
cefotaxime                       3844
levofloxacin                     3707
cefuroxime                       3552
ampicillin-sulbactam             3332
cefazolin                        3271
tigecycline                      3059
colistin                         2990
tetracycline                     2891
ertapenem                        2777
amoxicillin-clavulanic acid      2332
pentizidone                      2317
azithromycin            